# Image Staging Check

Build the image inventory from the frozen pilot manifests and check only the configured staging directory.
No archive is available, so extraction remains pending.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs/paths.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "configs/paths.yaml").is_file(), "Run from the project or notebooks directory"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.check_missing_assets import (
    check_assets,
    default_image_directory,
    load_asset_manifest,
    summarize_assets,
    validate_image_path,
)

MANIFEST_ROOT = PROJECT_ROOT / "data/manifests"

In [2]:
query_file = MANIFEST_ROOT / "pilot_queries.parquet"
evidence_file = MANIFEST_ROOT / "pilot_evidence.parquet"
assert query_file.is_file() and evidence_file.is_file(), "Frozen manifests are required"

queries = pd.read_parquet(query_file, engine="pyarrow")
evidence = pd.read_parquet(evidence_file, engine="pyarrow")
assert len(queries) == 200
assert len(evidence) == 5000
assert {"query_image_path", "image_id", "image_source"}.issubset(queries.columns)
assert {"image_path", "id", "source"}.issubset(evidence.columns)
assert not queries["query_image_path"].isna().any()
assert not evidence["image_path"].isna().any()

## Asset Counts

In [3]:
query_paths = set(queries["query_image_path"])
evidence_paths = set(evidence["image_path"])
all_paths = query_paths | evidence_paths
shared_paths = query_paths & evidence_paths

counts = {
    "query path rows": len(queries),
    "unique query paths": len(query_paths),
    "evidence path rows": len(evidence),
    "unique evidence paths": len(evidence_paths),
    "shared paths": len(shared_paths),
    "total required images": len(all_paths),
}
pd.Series(counts, name="count").to_frame()

,count
query path rows,200
unique query paths,200
evidence path rows,5000
unique evidence paths,5000
shared paths,200
total required images,5000


## Path Checks

Keep each original metadata path unchanged. `normalized_archive_path` currently equals `image_path`.
Actual archive headers have not been checked; explicit CLI prefix options handle a confirmed layout difference.

In [4]:
malformed_paths = []
for image_path in sorted(all_paths):
    try:
        validate_image_path(image_path)
    except ValueError as error:
        malformed_paths.append({"image_path": image_path, "error": str(error)})

path_checks = {
    "null query paths": int(queries["query_image_path"].isna().sum()),
    "null evidence paths": int(evidence["image_path"].isna().sum()),
    "repeated query paths": len(queries) - len(query_paths),
    "repeated evidence paths": len(evidence) - len(evidence_paths),
    "malformed unique paths": len(malformed_paths),
}
display(pd.Series(path_checks, name="count").to_frame())
if malformed_paths:
    display(pd.DataFrame(malformed_paths))
    raise ValueError("Resolve malformed metadata paths before freezing the inventory")

,count
null query paths,0
null evidence paths,0
repeated query paths,0
repeated evidence paths,0
malformed unique paths,0


In [5]:
metadata_by_path = {}
for row in evidence.itertuples(index=False):
    metadata = (int(row.id), row.source)
    previous = metadata_by_path.get(row.image_path)
    assert previous is None or previous == metadata, "Conflicting evidence metadata"
    metadata_by_path[row.image_path] = metadata

for row in queries.itertuples(index=False):
    metadata = (int(row.image_id), row.image_source)
    previous = metadata_by_path.get(row.query_image_path)
    assert previous is None or previous == metadata, "Query/evidence image metadata disagrees"
    metadata_by_path[row.query_image_path] = metadata

In [6]:
asset_rows = []
for image_path in sorted(all_paths):
    is_query_image = image_path in query_paths
    is_evidence_image = image_path in evidence_paths
    if is_query_image and is_evidence_image:
        asset_role = "both"
    elif is_query_image:
        asset_role = "query_only"
    else:
        asset_role = "evidence_only"

    metadata_id, source = metadata_by_path[image_path]
    assert source == validate_image_path(image_path).parts[2], "Publisher disagrees with path"
    asset_rows.append({
        "image_path": image_path,
        "normalized_archive_path": image_path,
        "is_query_image": is_query_image,
        "is_evidence_image": is_evidence_image,
        "asset_role": asset_role,
        "source": source,
        "metadata_id": metadata_id,
    })

assets = pd.DataFrame(asset_rows)
role_counts = assets["asset_role"].value_counts()
role_counts = role_counts.reindex(["query_only", "evidence_only", "both"], fill_value=0)
display(role_counts.rename("images").to_frame())
display(assets["source"].value_counts().sort_index().rename("images").to_frame())
assets.head(3)

,images
asset_role,
query_only,0
evidence_only,4800
both,200


,images
source,
bbc,1250
guardian,1250
usa_today,1250
washington_post,1250


,image_path,normalized_archive_path,is_query_image,is_evidence_image,asset_role,source,metadata_id
0,visual_news/origin/bbc/images/0000/898.jpg,visual_news/origin/bbc/images/0000/898.jpg,False,True,evidence_only,bbc,1447857
1,visual_news/origin/bbc/images/0002/512.jpg,visual_news/origin/bbc/images/0002/512.jpg,False,True,evidence_only,bbc,1453689
2,visual_news/origin/bbc/images/0002/584.jpg,visual_news/origin/bbc/images/0002/584.jpg,False,True,evidence_only,bbc,1453945


In [7]:
asset_file = MANIFEST_ROOT / "pilot_image_assets.parquet"
if asset_file.exists():
    existing_assets = load_asset_manifest(asset_file)
    pd.testing.assert_frame_equal(existing_assets, assets)
else:
    assets.to_parquet(asset_file, engine="pyarrow", index=False)

reloaded_assets = load_asset_manifest(asset_file)
pd.testing.assert_frame_equal(reloaded_assets, assets)
assert len(reloaded_assets) == len(all_paths)
assert reloaded_assets["image_path"].is_unique
assert not reloaded_assets["image_path"].isna().any()

manifest_paths = set(reloaded_assets["image_path"])
assert query_paths.issubset(manifest_paths)
assert evidence_paths.issubset(manifest_paths)
coverage = {
    "manifest rows": len(reloaded_assets),
    "unique paths": len(manifest_paths),
    "query coverage (%)": 100 * len(query_paths & manifest_paths) / len(query_paths),
    "evidence coverage (%)": 100 * len(evidence_paths & manifest_paths) / len(evidence_paths),
}
pd.Series(coverage, name="value").to_frame()

,value
manifest rows,5000.0
unique paths,5000.0
query coverage (%),100.0
evidence coverage (%),100.0


## Availability

Paths are checked below the configured `pilot_root/images`, preserving their full metadata-relative hierarchy.
Missing local files do not imply missing files in VisualNews.

In [8]:
image_directory = default_image_directory()
print("Staging directory:", image_directory)
results = check_assets(reloaded_assets, image_directory)
summary, duplicate_hashes = summarize_assets(results)
availability_keys = ["expected_images", "found_images", "missing_local_images", "available_pct"]
pd.Series(summary).loc[availability_keys].rename("value").to_frame()

Staging directory: /Users/nirajmac/Documents/MajorProject/data/pilot/images


,value
expected_images,5000.0
found_images,0.0
missing_local_images,5000.0
available_pct,0.0


## Image Health

In [9]:
health_keys = [
    "readable_images", "corrupt_or_unreadable_images", "zero_byte_images",
    "unexpected_formats", "unsafe_staged_paths", "not_regular_files",
]
display(pd.Series(summary).loc[health_keys].rename("count").to_frame())
readable_images = results[results["readable"]]
if readable_images.empty:
    print("No readable staged images. Format and dimension checks remain pending.")
else:
    display(readable_images["format"].value_counts().rename("images").to_frame())
    display(readable_images[["width", "height"]].describe())

,count
readable_images,0.0
corrupt_or_unreadable_images,0.0
zero_byte_images,0.0
unexpected_formats,0.0
unsafe_staged_paths,0.0
not_regular_files,0.0


No readable staged images. Format and dimension checks remain pending.


## Duplicates

In [10]:
hash_keys = ["hashed_files", "unique_hashes", "duplicate_hash_groups", "files_in_duplicate_groups"]
display(pd.Series(summary).loc[hash_keys].rename("count").to_frame())
if summary["hashed_files"] == 0:
    print("No staged files were hashed; exact duplicate checks remain pending.")
elif not duplicate_hashes.empty:
    duplicate_paths = results[results["sha256"].isin(duplicate_hashes.index)]
    display(duplicate_paths[["sha256", "image_path"]].head(20))
print("SHA-256 checks exact binary duplicates, not visual near-duplicates.")

,count
hashed_files,0.0
unique_hashes,0.0
duplicate_hash_groups,0.0
files_in_duplicate_groups,0.0


No staged files were hashed; exact duplicate checks remain pending.
SHA-256 checks exact binary duplicates, not visual near-duplicates.


## Notes

The manifest and CLI scripts are prepared. Archive membership, extraction, and real image health remain pending.
No plots are needed for an empty staging directory. No embeddings or retrieval are run here.